# Satellite Intelligence Assignment

## BRONZE LAYER — DATA INGESTION

## Step 1 — Load the CSV Files

In [0]:
# Load input datasets
readings_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_readings.csv", header=True, inferSchema=True)

metadata_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_metadata.csv", header=True,inferSchema=True)


readings_df.display()
metadata_df.display()

## Check schema

In [0]:
# Print schema for parcel_readings
readings_df.printSchema()

# Print schema for parcel_metadata
metadata_df.printSchema()

## preview data


In [0]:
# Show sample rows from parcel_readings
readings_df.show(5)

# Show sample rows from parcel_metadata
metadata_df.show(5)

## Count rows

In [0]:
print("Total rows in parcel_readings:", readings_df.count())

print("Total rows in parcel_metadata:", metadata_df.count())

## Data Quality Audit

## NULL VALUE AUDIT

## For parcel_readings

In [0]:
from pyspark.sql.functions import col, count, when

In [0]:
readings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in readings_df.columns
]).show()

## For parcel metadata

In [0]:
metadata_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in metadata_df.columns
]).show()

## Duplicate check

## For parcel_readings

In [0]:
duplicate_readings = readings_df.groupBy("parcel_id", "date") \
    .count() \
    .filter(col("count") > 1)

duplicate_readings.show()

## For parcel_metadata

In [0]:
duplicate_metadata = metadata_df.groupBy("parcel_id") \
    .count() \
    .filter(col("count") > 1)

duplicate_metadata.show()

## Invalid NDVI Values

In [0]:
invalid_ndvi = readings_df.filter(
    (col("ndvi_value") < -1) |
    (col("ndvi_value") > 1)
)

print("Invalid NDVI count:", invalid_ndvi.count())

invalid_ndvi.show()

## Invalid Temperature Values

In [0]:
readings_df.describe(["temperature_c"]).show()

In [0]:
readings_df.filter(
    (col("temperature_c") < -20) |
    (col("temperature_c") > 60)
).show()

## Negative Rainfall

In [0]:
negative_rainfall = readings_df.filter(
    col("rainfall_mm") < 0
)

print("Negative rainfall rows:", negative_rainfall.count())
negative_rainfall.show()

## Invalid Sensor Status

In [0]:
readings_df.select("sensor_status").distinct().show()

## Invalid Dates

In [0]:
from pyspark.sql.functions import col, trim, try_to_date

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

readings_df = spark.read.csv(
    "/Volumes/satellite_intelligence/default/bronze/parcel_readings.csv",
    header=True,
    inferSchema=True
)

df = readings_df.withColumn("date_str", trim(col("date")))

# Safe parsing (no error, returns NULL for invalid formats)
df = df.withColumn(
    "parsed_date",
    try_to_date(col("date_str"), "yyyy-MM-dd")
)

# Data Quality Audit
invalid_dates_df = df.filter(col("parsed_date").isNull())
valid_dates_df = df.filter(col("parsed_date").isNotNull())

print("Total Records:", df.count())
print("Valid Dates:", valid_dates_df.count())
print("Invalid Dates:", invalid_dates_df.count())

invalid_dates_df.show(truncate=False)

In [0]:
readings_df.join(
    metadata_df,
    "parcel_id",
    "left_anti"
).show()

In [0]:
metadata_df.filter(col("area_hectares") <= 0).show()

## Task 2: Build a Clean Pipeline
## 1. Understanding the Requirement


## Read CSVs
## Clean datasets
## Join datasets
## Create final time-series dataset
## Export cleaned CSV

## SILVER LAYER — CLEANING

In [0]:
# Load input datasets
readings_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_readings.csv", header=True, inferSchema=True)

metadata_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_metadata.csv", header=True,inferSchema=True)


readings_df .display()
metadata_df.display()

## Date Standardization

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce, trim

# 1. Force Spark to use Legacy parsing (additional safety for Databricks)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 2. Load the datasets (Using your absolute paths)
readings_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_readings.csv", header=True, inferSchema=True)
metadata_df = spark.read.csv("/Volumes/satellite_intelligence/default/bronze/parcel_metadata.csv", header=True, inferSchema=True)

# 3. Process Readings Data
# We use try_to_date so it returns NULL instead of crashing on a mismatch
readings_df = readings_df.withColumn(
    "date",
    coalesce(
        try_to_date(trim(col("date")), "yyyy-MM-dd"),   # Handles 2026-01-27
        try_to_date(trim(col("date")), "dd/MM/yyyy"),   # Handles 16/05/2026
        try_to_date(trim(col("date")), "dd-MMM-yyyy")   # Handles 20-Jan-2026
    )
)

# 4. Process Metadata Data (for consistency)
metadata_df = metadata_df.withColumn(
    "sowing_date",
    coalesce(
        try_to_date(trim(col("sowing_date")), "yyyy-MM-dd"),
        try_to_date(trim(col("sowing_date")), "dd/MM/yyyy"),
        try_to_date(trim(col("sowing_date")), "dd-MMM-yyyy")
    )
)

# 5. Display the final cleaned data
readings_df.display()
metadata_df.display()

In [0]:
from pyspark.sql.functions import to_date, col

In [0]:
metadata_df = metadata_df.withColumn(
    "sowing_date",
    to_date(col("sowing_date"), "yyyy-MM-dd")
)

## Remove Duplicates

In [0]:
readings_df = readings_df.dropDuplicates(
    ["parcel_id", "date"]
)

## NDVI Cleaning

In [0]:
readings_df = readings_df.withColumn(
    "ndvi_value",
    when(
        (col("ndvi_value") >= -1) &
        (col("ndvi_value") <= 1),
        col("ndvi_value")
    ).otherwise(None)
)

## Sensor Status

In [0]:
from pyspark.sql.functions import lower, trim, col

In [0]:

readings_df = readings_df.withColumn(
    "sensor_status",
    lower(trim(col("sensor_status")))
)

## Weather Cleaning

In [0]:
readings_df = readings_df.withColumn(
    "rainfall_mm",
    when(
        col("rainfall_mm") >= 0,
        col("rainfall_mm")
    ).otherwise(None)
)

In [0]:
readings_df = readings_df.withColumn(
    "temperature_c",
    when(
        (col("temperature_c") >= -10) &
        (col("temperature_c") <= 60),
        col("temperature_c")
    ).otherwise(None)
)

## remove invalid keys

In [0]:
readings_df = readings_df.filter(
    col("parcel_id").isNotNull()
)

metadata_df = metadata_df.filter(
    col("parcel_id").isNotNull()
)

In [0]:
metadata_df = metadata_df.filter(col("area_hectares") > 0)

## Join on data frames

In [0]:
final_df = readings_df.join(
    metadata_df,
    on="parcel_id",
    how="inner"
)

In [0]:
final_df.show()

In [0]:
final_df.coalesce(1).write.mode("overwrite").option(
    "header",
    True
).csv("/Volumes/satellite_intelligence/default/silver/cleaned_parcel_timeseries.csv")

In [0]:
from pyspark.sql.functions import col, trim, upper
good_df = final_df.filter(
    upper(trim(col("sensor_status"))) == "OK"
)
good_df.show()

In [0]:
from pyspark.sql.functions import datediff, col

good_df = good_df.withColumn(
    "days_from_sowing",
    datediff(col("date"), col("sowing_date"))
)

good_df.show()


In [0]:
before_df = good_df.filter(
    (col("days_from_sowing") >= -30) &
    (col("days_from_sowing") < 0)
)



In [0]:
after_df = good_df.filter(
    (col("days_from_sowing") > 0) &
    (col("days_from_sowing") <= 30)
)

In [0]:
from pyspark.sql.functions import avg
before_avg = before_df.groupBy("crop_type").agg(
    avg("ndvi_value").alias("mean_ndvi_before")
)

In [0]:
from pyspark.sql.functions import countDistinct
after_avg = after_df.groupBy("crop_type").agg(
    avg("ndvi_value").alias("mean_ndvi_after"),
    countDistinct("parcel_id").alias("n_parcels")
)

In [0]:
analysis_df = before_avg.join(
    after_avg,
    on="crop_type",
    how="inner"
)

print("Crop-wise NDVI Analysis")
analysis_df.show()

In [0]:
# Store Gold analytical dataset as CSV

analysis_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/satellite_intelligence/default/gold/crop_ndvi_analysis.csv")